# Stack with simple average meta
Same base models as 98 (LightGBM + LSTM + Ridge + EWM) trained once on full pooled data; blend = simple average of the four predictions (no learned meta-learner). For comparison with 98-xgb-lstm-stack-pool.

In [ ]:
import sys
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
import lightgbm as lgb

REPO_ROOT = Path.cwd().parent.parent
BACKEND_DIR = REPO_ROOT / "backend"
sys.path.insert(0, str(BACKEND_DIR))
sys.path.insert(0, str(Path.cwd()))

from _pool_common import (
    load_pool_data,
    build_pooled_train_stack,
    compute_metrics_averaged_over_windows,
    metrics_to_parquet,
    fetch_cnn_fear_greed_index,
    TEST_SIZE,
    FORECAST_HORIZON,
    ROLLING_STEP,
    MIN_TRAIN_STACK,
    ARTIFACTS_DIR,
    TICKERS,
)

import json
# Load best params from 05/06 random search if available (run 05 and 06 first to generate artifacts)
_lgb_path = ARTIFACTS_DIR / "best_lgb_params.json"
if _lgb_path.exists():
    with open(_lgb_path) as f:
        LGB_PARAMS = json.load(f)
    LGB_PARAMS.setdefault("random_state", 42)
    LGB_PARAMS.setdefault("verbosity", -1)
else:
    LGB_PARAMS = dict(n_estimators=100, max_depth=4, learning_rate=0.01, random_state=42, verbosity=-1)
_ridge_path = ARTIFACTS_DIR / "best_ridge_params.json"
if _ridge_path.exists():
    with open(_ridge_path) as f:
        _ridge = json.load(f)
    RIDGE_ALPHA = _ridge["alpha"]
else:
    RIDGE_ALPHA = 1.0
EWM_SPAN = 20

LAG_RETURNS = 5
RSI_PERIOD = 14
MACD_FAST, MACD_SLOW, MACD_SIGNAL = 12, 26, 9
SEQ_LEN = 30
LSTM_EPOCHS_FINAL = 100


In [ ]:
def _rsi(series: pd.Series, period: int) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.ewm(alpha=1 / period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False).mean()
    rs = avg_gain / np.where(avg_loss != 0, avg_loss, 1e-10)
    return 100 - (100 / (1 + rs))


def build_feature_df(grp: pd.DataFrame):
    df = grp.sort_values("timestamp").copy()
    df["close"] = df["close"].astype(float)
    df["return"] = df["close"].pct_change()
    for i in range(1, LAG_RETURNS + 1):
        df[f"ret_lag_{i}"] = df["return"].shift(i)
    if "volume" in df.columns:
        df["volume_lag_1"] = df["volume"].astype(float).shift(1)
    else:
        df["volume_lag_1"] = np.nan
    df["rsi"] = _rsi(df["close"], RSI_PERIOD)
    ema_fast = df["close"].ewm(span=MACD_FAST, adjust=False).mean()
    ema_slow = df["close"].ewm(span=MACD_SLOW, adjust=False).mean()
    df["macd_line"] = ema_fast - ema_slow
    df["macd_signal"] = df["macd_line"].ewm(span=MACD_SIGNAL, adjust=False).mean()
    if "vix" in df.columns:
        vix = df["vix"].astype(np.float64)
        df["vix_sma_5"] = vix.shift(1).rolling(5).mean()
        df["vix_velocity"] = vix.diff(1)
        df["vix_momentum"] = vix - df["vix_sma_5"]
    else:
        df["vix_sma_5"] = np.nan
        df["vix_velocity"] = np.nan
        df["vix_momentum"] = np.nan
    df["month"] = pd.to_datetime(df["timestamp"]).dt.month
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    if "fear_greed" not in df.columns:
        df["fear_greed"] = 50.0
    else:
        df["fear_greed"] = df["fear_greed"].fillna(50.0)
    df["fear_greed_lag_1"] = df["fear_greed"].shift(1)
    df["fear_greed_lag_5"] = df["fear_greed"].shift(5)
    df["fear_greed_change"] = df["fear_greed_lag_1"] - df["fear_greed_lag_5"]
    for h in range(1, FORECAST_HORIZON + 1):
        df[f"target_{h}"] = df["return"].shift(-h)
    feature_cols_lstm = [f"ret_lag_{i}" for i in range(1, LAG_RETURNS + 1)] + ["volume_lag_1", "rsi", "macd_line", "macd_signal"]
    feature_cols_xgb = ["vix_sma_5", "vix_velocity", "vix_momentum", "month_sin", "month_cos", "fear_greed_change"]
    target_cols = [f"target_{h}" for h in range(1, FORECAST_HORIZON + 1)]
    base_cols = ["timestamp", "close", "return"] + feature_cols_lstm + feature_cols_xgb + target_cols
    out = df[[c for c in base_cols if c in df.columns]].copy()
    return out.dropna(), feature_cols_lstm, feature_cols_xgb, target_cols


def build_sequences(X: np.ndarray, y: np.ndarray, seq_len: int):
    n = len(X)
    if n < seq_len + 1:
        return None, None
    X_seq, y_seq = [], []
    for i in range(seq_len, n):
        X_seq.append(X[i - seq_len : i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)

In [ ]:
try:
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense
    from tensorflow.keras.callbacks import EarlyStopping
    HAS_LSTM = True
except Exception:
    HAS_LSTM = False


def train_lstm(X_seq: np.ndarray, y_seq: np.ndarray, horizon: int, epochs=20, use_early_stopping=False):
    if not HAS_LSTM or X_seq is None or len(X_seq) < 10:
        return None
    n_feat = X_seq.shape[2]
    model = Sequential([
        LSTM(32, activation="tanh", input_shape=(X_seq.shape[1], n_feat)),
        Dense(horizon),
    ])
    model.compile(optimizer="adam", loss="mse")
    callbacks = [EarlyStopping(monitor="loss", patience=5, restore_best_weights=True)] if use_early_stopping and HAS_LSTM else []
    model.fit(X_seq, y_seq, epochs=epochs, callbacks=callbacks, verbose=0)
    return model

In [1]:
def train_global_stack_simple_avg(stacked: pd.DataFrame, horizon: int):
    """Train LightGBM + LSTM + Ridge once on full pooled data (no OOF, no meta-learner). EWM is computed at predict time. Returns dict for predict_stack_simple_avg."""
    pooled = build_pooled_train_stack(stacked, TEST_SIZE, MIN_TRAIN_STACK)
    if pooled.empty:
        return None
    feat_dfs = []
    for sym in pooled["symbol"].unique():
        grp = pooled[pooled["symbol"] == sym].copy()
        try:
            feat_df, feature_cols_lstm, feature_cols_xgb, target_cols = build_feature_df(grp)
        except Exception:
            continue
        if len(feat_df) < MIN_TRAIN_STACK + horizon:
            continue
        feat_df["_symbol_"] = sym
        feat_dfs.append(feat_df)
    if not feat_dfs:
        return None
    pooled_feat = pd.concat(feat_dfs, ignore_index=True)
    feature_cols_ridge = feature_cols_lstm + feature_cols_xgb
    y_full = pooled_feat[target_cols].values.astype(np.float32)
    scaler_xgb_by_sym = {}
    scaler_lstm_by_sym = {}
    scaler_ridge_by_sym = {}
    for sym in pooled_feat["_symbol_"].unique():
        m = pooled_feat["_symbol_"] == sym
        scaler_xgb_by_sym[sym] = StandardScaler().fit(pooled_feat.loc[m, feature_cols_xgb].values.astype(np.float32))
        scaler_lstm_by_sym[sym] = StandardScaler().fit(pooled_feat.loc[m, feature_cols_lstm].values.astype(np.float32))
        scaler_ridge_by_sym[sym] = StandardScaler().fit(pooled_feat.loc[m, feature_cols_ridge].values.astype(np.float64))
    X_xgb_full_s = np.empty((len(pooled_feat), len(feature_cols_xgb)), dtype=np.float32)
    X_lstm_full_s = np.empty((len(pooled_feat), len(feature_cols_lstm)), dtype=np.float32)
    X_ridge_full_s = np.empty((len(pooled_feat), len(feature_cols_ridge)), dtype=np.float64)
    for sym in pooled_feat["_symbol_"].unique():
        m = pooled_feat["_symbol_"] == sym
        X_xgb_full_s[m] = scaler_xgb_by_sym[sym].transform(pooled_feat.loc[m, feature_cols_xgb].values.astype(np.float32))
        X_lstm_full_s[m] = scaler_lstm_by_sym[sym].transform(pooled_feat.loc[m, feature_cols_lstm].values.astype(np.float32))
        X_ridge_full_s[m] = scaler_ridge_by_sym[sym].transform(pooled_feat.loc[m, feature_cols_ridge].values.astype(np.float64))
    lgb_multi = MultiOutputRegressor(lgb.LGBMRegressor(**LGB_PARAMS))
    lgb_multi.fit(X_xgb_full_s, y_full)
    X_seq_list, y_seq_list = [], []
    for sym in pooled_feat["_symbol_"].unique():
        sym_mask = pooled_feat["_symbol_"] == sym
        X_s = X_lstm_full_s[sym_mask]
        y_s = y_full[sym_mask]
        X_seq, y_seq = build_sequences(X_s, y_s, SEQ_LEN)
        if X_seq is not None and len(X_seq) >= 10:
            X_seq_list.append(X_seq)
            y_seq_list.append(y_seq)
    lstm_model = train_lstm(np.concatenate(X_seq_list, axis=0), np.concatenate(y_seq_list, axis=0), horizon, epochs=LSTM_EPOCHS_FINAL, use_early_stopping=True) if X_seq_list else None
    ridge_multi = MultiOutputRegressor(Ridge(alpha=RIDGE_ALPHA, random_state=42))
    ridge_multi.fit(X_ridge_full_s, y_full.astype(np.float64))
    return {
        "scaler_xgb_by_sym": scaler_xgb_by_sym, "scaler_lstm_by_sym": scaler_lstm_by_sym, "scaler_ridge_by_sym": scaler_ridge_by_sym,
        "lgb_multi": lgb_multi, "lstm_model": lstm_model, "ridge_multi": ridge_multi,
        "feature_cols_lstm": feature_cols_lstm, "feature_cols_xgb": feature_cols_xgb, "feature_cols_ridge": feature_cols_ridge,
        "target_cols": target_cols, "EWM_SPAN": EWM_SPAN,
    }


def predict_stack_simple_avg(context_df: pd.DataFrame, horizon: int, global_stack: dict) -> list:
    """Predict 21 price steps: simple average of LGB, LSTM, Ridge, and EWM base predictions."""
    if global_stack is None or global_stack.get("lgb_multi") is None:
        return []
    try:
        feat_df, feature_cols_lstm, feature_cols_xgb, _ = build_feature_df(context_df)
    except Exception:
        return []
    if len(feat_df) < SEQ_LEN + 1:
        return []
    feature_cols_ridge = global_stack.get("feature_cols_ridge", feature_cols_lstm + feature_cols_xgb)
    sym_d = global_stack.get("scaler_xgb_by_sym", {})
    symbol = context_df["symbol"].iloc[0] if "symbol" in context_df.columns and len(context_df) else (list(sym_d.keys())[0] if sym_d else None)
    if symbol is None:
        return []
    scaler_xgb = sym_d.get(symbol, list(sym_d.values())[0]) if sym_d else None
    scaler_lstm = global_stack.get("scaler_lstm_by_sym", {}).get(symbol, list(global_stack.get("scaler_lstm_by_sym", {}).values())[0]) if global_stack.get("scaler_lstm_by_sym") else None
    scaler_ridge = global_stack.get("scaler_ridge_by_sym", {}).get(symbol, list(global_stack.get("scaler_ridge_by_sym", {}).values())[0]) if global_stack.get("scaler_ridge_by_sym") else None
    if scaler_xgb is None or scaler_lstm is None or scaler_ridge is None:
        return []
    lgb_multi = global_stack["lgb_multi"]
    lstm_model = global_stack["lstm_model"]
    ridge_multi = global_stack["ridge_multi"]
    EWM_SPAN = global_stack.get("EWM_SPAN", 20)
    X_xgb_s = scaler_xgb.transform(feat_df[feature_cols_xgb].values.astype(np.float32))
    X_lstm_s = scaler_lstm.transform(feat_df[feature_cols_lstm].values.astype(np.float32))
    X_ridge_s = scaler_ridge.transform(feat_df[feature_cols_ridge].values.astype(np.float64))
    last_idx = len(feat_df) - 1
    xgb_21 = lgb_multi.predict(X_xgb_s[last_idx : last_idx + 1]).ravel()
    ridge_21 = ridge_multi.predict(X_ridge_s[last_idx : last_idx + 1]).ravel()
    ret_series = feat_df["return"].values.astype(np.float64)
    if len(ret_series) >= EWM_SPAN:
        ewm_val = float(pd.Series(ret_series).ewm(span=EWM_SPAN).mean().iloc[-1])
        ewm_21 = np.full(horizon, ewm_val, dtype=np.float32)
    else:
        ewm_21 = np.full(horizon, float(np.nanmean(ret_series)) if len(ret_series) else 0.0, dtype=np.float32)
    if lstm_model is not None and last_idx >= SEQ_LEN:
        seq = X_lstm_s[last_idx - SEQ_LEN : last_idx].reshape(1, SEQ_LEN, -1)
        lstm_21 = lstm_model.predict(seq, verbose=0).ravel()
    else:
        lstm_21 = xgb_21
    final_returns = (np.array(xgb_21) + np.array(lstm_21) + np.array(ridge_21) + np.array(ewm_21)) / 4.0
    p0 = float(context_df["close"].iloc[-1])
    prices = p0 * np.cumprod(np.concatenate([[1.0], 1.0 + final_returns]))[1:]
    return [float(p) for p in prices[:horizon]]

NameError: name 'pd' is not defined

In [ ]:
stacked = load_pool_data(with_vix=True, with_volume=True)
symbol_start = pd.to_datetime(stacked["timestamp"]).min().strftime("%Y-%m-%d")
fear_greed_df = fetch_cnn_fear_greed_index(start_date=symbol_start)
if not fear_greed_df.empty:
    stacked["date"] = pd.to_datetime(stacked["timestamp"]).dt.normalize()
    fear_greed_df["date"] = pd.to_datetime(fear_greed_df["timestamp"]).dt.normalize()
    stacked = stacked.merge(fear_greed_df[["date", "fear_greed"]], on="date", how="left")
    stacked["fear_greed"] = stacked["fear_greed"].ffill().bfill()
    stacked = stacked.drop(columns=["date"])
print(stacked.groupby("symbol").size())
stacked.head(10)

In [ ]:
# Train base models once on full pooled data (no OOF, no meta-learner)
global_stack = train_global_stack_simple_avg(stacked, FORECAST_HORIZON)
pooled_train = build_pooled_train_stack(stacked, TEST_SIZE, MIN_TRAIN_STACK)
print("Simple-avg stack trained on", len(pooled_train), "pooled train rows.")

In [ ]:
# Backtest: same 60-day test window, 21-day horizon, step 7 days
model_name = "stack_simple_avg"
all_preds = []
for sym in TICKERS:
    grp = stacked[stacked["symbol"] == sym].copy()
    if grp.empty:
        continue
    grp = grp.sort_values("timestamp").reset_index(drop=True)
    prices = grp.set_index("timestamp")["close"].astype(float).dropna()
    n = len(prices)
    if n < TEST_SIZE + MIN_TRAIN_STACK:
        continue
    split_idx = n - TEST_SIZE
    test_index = prices.index[split_idx:]
    test_values = prices.values[split_idx:]
    preds = []
    window_ix = 0
    start = 0
    while start + FORECAST_HORIZON <= TEST_SIZE:
        context_cols = ["timestamp", "close", "vix", "symbol"] + [c for c in ["volume", "fear_greed"] if c in grp.columns]
        context_df = grp.iloc[: split_idx + start][context_cols].copy()
        if len(context_df) < MIN_TRAIN_STACK:
            start += ROLLING_STEP
            continue
        price_list = predict_stack_simple_avg(context_df, FORECAST_HORIZON, global_stack)
        if not price_list or len(price_list) < FORECAST_HORIZON:
            start += ROLLING_STEP
            window_ix += 1
            continue
        for h in range(FORECAST_HORIZON):
            idx = start + h
            ts = test_index[idx]
            y_true = float(test_values[idx])
            y_pred = float(price_list[h])
            preds.append({"timestamp": ts, "y_true": y_true, "y_pred": y_pred, "window_ix": window_ix})
        window_ix += 1
        start += ROLLING_STEP
    if preds:
        pred_df = pd.DataFrame(preds)
        pred_df["symbol"] = sym
        all_preds.append(pred_df)

pred_stack = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame(columns=["timestamp", "y_true", "y_pred", "window_ix", "symbol"])
print(pred_stack.groupby("symbol").size() if not pred_stack.empty else "No predictions.")
pred_stack.head()

In [ ]:
metrics_rows = []
for sym in pred_stack["symbol"].unique():
    sub = pred_stack[pred_stack["symbol"] == sym]
    m = compute_metrics_averaged_over_windows(sub)
    metrics_rows.append({"model": model_name, "symbol": sym, **m})
m_overall = compute_metrics_averaged_over_windows(pred_stack)
metrics_rows.append({"model": model_name, "symbol": "overall", **m_overall})

metrics_df = pd.DataFrame(metrics_rows)
print(metrics_df.to_string())
metrics_to_parquet(metrics_rows, ARTIFACTS_DIR / "metrics_stack_simple_avg_pool.parquet")
print("Saved:", ARTIFACTS_DIR / "metrics_stack_simple_avg_pool.parquet")